# Consistency-regularised multi-specialist grokking

Pilot sweep for the experiment described in `reports/consistency_regularized_grokking_plan.md`.

Each cell of the sweep is one full multi-specialist run with a different consistency configuration. The pilot is 4 cells, intended to fit comfortably within Kaggle's GPU session limit:

| cell | lambda | warmup | what it tests |
|---|---|---|---|
| `lam0_baseline` | 0 | 0 | multi-specialist baseline, no consistency |
| `lam1_constant` | 1.0 | 0 | failure-mode-2 probe (memorize-before-align) |
| `lam1_warmup1k` | 1.0 | 1000 | headline cell (delayed alignment) |
| `lam10_warmup1k` | 10.0 | 1000 | strong consistency, see if structure forms |

All cells share: M=4 specialists, disjoint shards, train_data_pct=50, max_lr=5e-4, weight_decay=0.1, 25k steps. Per-cell artifacts go to `/kaggle/working/consistency_runs/pilot_v1/<cell_name>/`.

**Before running:** make sure the runtime is set to GPU (T4 or P100) and that internet is enabled (for the initial git clone).

## 1. Clone the repo

Pinned to commit `f4b86944` (this revision implements `grok/consistency_training.py`). Re-run with a different `GROK_COMMIT` if you want to test a different revision.

In [ ]:
import os, subprocess, sys

GROK_COMMIT = os.environ.get('GROK_COMMIT', 'f4b86944deeda3af577477c8ec24133b81ea0cb5')
GROK_REPO = os.environ.get('GROK_REPO', 'https://github.com/yazankb/grok.git')
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/grok_work')
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, 'grok')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', GROK_REPO, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', GROK_COMMIT], check=True)
print('repo at:', REPO_DIR)
print('commit :', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())

## 2. Install dependencies

Kaggle's GPU base image already has torch and pytorch_lightning. We need `mod` (used by `grok/data.py` for modular arithmetic) and a recent `sympy`.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mod', 'sympy'], check=True)
import torch, pytorch_lightning as pl
print('torch =', torch.__version__, '| lightning =', pl.__version__, '| cuda =', torch.cuda.is_available())

## 3. Run the pilot sweep

On a T4 GPU each cell runs ~10–20 minutes (4 specialists × 25k steps), so the full pilot is ~1–1.5 hours. The script writes incremental progress to `sweep_summary.json` so partial results survive interrupted sessions.

If you want to do a faster sanity check first, set `STEPS=2000` in the env.

In [ ]:
STEPS = int(os.environ.get('STEPS', 25000))
SWEEP_NAME = os.environ.get('SWEEP_NAME', 'pilot_v1')
LOGDIR = os.environ.get('GROK_LOGDIR', os.path.join(WORK, 'consistency_runs'))
GPU = '0' if torch.cuda.is_available() else '-1'

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + ':' + env.get('PYTHONPATH', '')

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, 'scripts', 'run_consistency_sweep.py'),
    '--sweep_name', SWEEP_NAME,
    '--logdir', LOGDIR,
    '--consistency_steps', str(STEPS),
    '--gpu', GPU,
    '--n_models', '4',
    '--sharding', 'disjoint',
    '--train_data_pct', '50',
    '--max_lr', '5e-4',
    '--weight_decay', '0.1',
    '--shard_batch_size', '256',
    '--consistency_batch_size', '256',
    '--eval_every', '500',
    '--checkpoint_every', '5000',
    '--log_every', '50',
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env, cwd=REPO_DIR)

## 4. Inspect results

Read the sweep summary and print a small comparison table.

In [ ]:
import json

summary_path = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_summary.json')
with open(summary_path) as f:
    summary = json.load(f)

print(f"sweep: {summary['sweep_name']}  ({len(summary['cells'])} cell(s))")
print(f"started: {summary.get('started_at')}  finished: {summary.get('finished_at')}")
print()
fmt = '{:<22} {:<8} {:>10} {:>12} {:>10}'
print(fmt.format('cell', 'status', 'best_ens', 'best_merged', 'sec'))
print('-' * 64)
for c in summary['cells']:
    r = c.get('results', {}) or {}
    print(fmt.format(
        c['name'], c['status'],
        f"{r.get('best_val_acc_ensemble', float('nan')):.2f}" if 'best_val_acc_ensemble' in r else 'n/a',
        f"{r.get('best_val_acc_merged', float('nan')):.2f}" if 'best_val_acc_merged' in r else 'n/a',
        f"{c.get('elapsed_sec', 0):.0f}",
    ))

## 5. Plot per-cell training curves

Loads `consistency_eval.csv` from each cell and plots:

1. ensemble val acc vs step (headline metric)
2. merged val acc vs step (weight-average behaviour)
3. pairwise KL on val (alignment diagnostic)
4. unsup output entropy (collapse diagnostic)

In [ ]:
import csv
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
(ax_ens, ax_merged), (ax_kl, ax_ent) = axes

for c in summary['cells']:
    if c['status'] != 'ok':
        continue
    eval_csv = os.path.join(c['experiment_dir'], 'consistency_eval.csv')
    if not os.path.isfile(eval_csv):
        continue
    with open(eval_csv) as f:
        rows = list(csv.DictReader(f))
    steps = [int(r['step']) for r in rows]
    ax_ens.plot(steps, [float(r['val_acc_ensemble']) for r in rows], label=c['name'])
    ax_merged.plot(steps, [float(r['val_acc_merged']) for r in rows], label=c['name'])
    ax_kl.plot(steps, [float(r['pairwise_kl_val_mean']) for r in rows], label=c['name'])
    ax_ent.plot(steps, [float(r['unsup_entropy_mean']) for r in rows], label=c['name'])

for ax, title in [
    (ax_ens, 'Ensemble val acc (%)'),
    (ax_merged, 'Merged val acc (%)'),
    (ax_kl, 'Pairwise val KL (alignment)'),
    (ax_ent, 'Unsup output entropy (collapse diag)'),
]:
    ax.set_title(title)
    ax.set_xlabel('step')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
ax_kl.set_yscale('log')
fig.tight_layout()
out_png = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_curves.png')
fig.savefig(out_png, dpi=120)
print('saved:', out_png)
plt.show()

## 6. Bundle artifacts for download

Zips the whole sweep dir so you can pull it off Kaggle in one click.

In [ ]:
import shutil
sweep_dir = os.path.join(LOGDIR, SWEEP_NAME)
out_zip = os.path.join(WORK, f'{SWEEP_NAME}.zip')
if os.path.isfile(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip[:-4], 'zip', root_dir=sweep_dir)
print('bundle:', out_zip, '|', os.path.getsize(out_zip) / 1e6, 'MB')